# 01 · Convertir fuentes a modelo canónico

Esta libreta convierte los archivos de las 8 fuentes/editoriales ubicados en `01_recoleccion_manual/03_descargas_raw/` al modelo canónico del proyecto.

**Salida:** 8 archivos CSV, uno por fuente, en `03_modelo_canonico/`.

Esta libreta **no genera todavía el archivo único integrado**. La unión se hará después de la modificación manual pendiente.

Columnas finales, exactamente en este orden:

```text
indice
Titulo
Año
Autor_norm
Afiliacion1
Afiliacion2
ISBN
ISSN
Doi
URL
Area
Subarea
Keywords
Abstract
```

Reglas aplicadas:

- No inventar DOI, ISBN, ISSN, afiliaciones, keywords, abstracts ni subáreas.
- `Afiliacion2` queda vacía en todos los archivos.
- Si una fuente ya traía información en `Afiliacion2`, se integra en `Afiliacion1` para no perderla.
- Los autores se colocan en `Autor_norm` usando la columna más completa disponible.
- DOI, ISBN e ISSN se llenan solo desde columnas equivalentes o muy cercanas. No se agrega `eISSN` al ISSN.
- `Subarea` queda vacía porque no hay evidencia clara y homogénea para asignarla.
- `Area` solo puede ser `ISBD`, `CC`, `IA`, `TC`, `SIAV`, `RS`.

## 1. Configuración general

La ruta esperada del proyecto es `C:\Users\hazar\Desktop\PROYECTO`. Si ejecutas la libreta desde `PROYECTO/notebooks`, la raíz se detecta sola.

El código usa primero `01_recoleccion_manual/03_descargas_raw`.

In [1]:
from pathlib import Path
import csv
import re
import sys
import unicodedata
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

try:
    csv.field_size_limit(sys.maxsize)
except OverflowError:
    csv.field_size_limit(2_147_483_647)

COLUMNAS_CANONICAS = [
    "indice", "Titulo", "Año", "Autor_norm", "Afiliacion1", "Afiliacion2",
    "ISBN", "ISSN", "Doi", "URL", "Area", "Subarea", "Keywords", "Abstract",
]

AREAS_VALIDAS = {"ISBD", "CC", "IA", "TC", "SIAV", "RS"}
PERIODO = "2024-03_2025-12"

# Cambia a False si se quiere conservar índices originales de fuentes.
REGENERAR_INDICE = True

# La salida principal es CSV.
GUARDAR_XLSX = False

RUTA_PROYECTO_RESPALDO = Path(r"C:\Users\hazar\Desktop\PROYECTO")

FUENTES = {
    "acm": {
        "carpetas": ["acm"],
        "prefijos_archivo": ["acm"],
        "prefijo_indice": "ACM",
        "salida": f"acm_canonico_{PERIODO}.csv",
    },
    "ebsco": {
        "carpetas": ["ebsco"],
        "prefijos_archivo": ["ebsco"],
        "prefijo_indice": "EBSCO",
        "salida": f"ebsco_canonico_{PERIODO}.csv",
    },
    "engineering_village": {
        "carpetas": ["engineering_village", "ev"],
        "prefijos_archivo": ["ev", "engineering_village"],
        "prefijo_indice": "EV",
        "salida": f"engineering_village_canonico_{PERIODO}.csv",
    },
    "ieee_xplore": {
        "carpetas": ["ieee_xplore", "ieee"],
        "prefijos_archivo": ["ieee"],
        "prefijo_indice": "IEEE",
        "salida": f"ieee_xplore_canonico_{PERIODO}.csv",
    },
    "proquest": {
        "carpetas": ["proquest"],
        "prefijos_archivo": ["proquest"],
        "prefijo_indice": "PROQUEST",
        "salida": f"proquest_canonico_{PERIODO}.csv",
    },
    "sciencedirect": {
        "carpetas": ["sciencedirect", "science_direct"],
        "prefijos_archivo": ["sciencedirect", "science_direct"],
        "prefijo_indice": "SD",
        "salida": f"sciencedirect_canonico_{PERIODO}.csv",
    },
    "scopus": {
        "carpetas": ["scopus"],
        "prefijos_archivo": ["scopus"],
        "prefijo_indice": "SCOPUS",
        "salida": f"scopus_canonico_{PERIODO}.csv",
    },
    "web_of_science": {
        "carpetas": ["web_of_science", "wos"],
        "prefijos_archivo": ["wos", "web_of_science"],
        "prefijo_indice": "WOS",
        "salida": f"web_of_science_canonico_{PERIODO}.csv",
    },
}


def localizar_raiz_proyecto():
    """Busca la raíz del proyecto usando la carpeta 01_recoleccion_manual como marcador."""
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents, RUTA_PROYECTO_RESPALDO]
    revisados = set()
    for ruta in candidatos:
        ruta = Path(ruta)
        if ruta in revisados:
            continue
        revisados.add(ruta)
        if (ruta / "01_recoleccion_manual").exists():
            return ruta.resolve()
    raise FileNotFoundError(
        "No pude localizar la raíz del proyecto. Ejecuta la libreta desde PROYECTO/notebooks "
        "o ajusta RUTA_PROYECTO_RESPALDO."
    )


PROYECTO = localizar_raiz_proyecto()
RAW_CANDIDATOS = [
    PROYECTO / "01_recoleccion_manual" / "03_descargas_raw",
    PROYECTO / "01_recoleccion_manual" / "04_descargas_raw",
]
RAW_DIR = next((ruta for ruta in RAW_CANDIDATOS if ruta.exists()), None)
if RAW_DIR is None:
    raise FileNotFoundError("No encontré 03_descargas_raw ni 04_descargas_raw dentro de 01_recoleccion_manual.")

OUT_DIR = PROYECTO / "02_modelo canonico"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR = PROYECTO / "outputs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", PROYECTO)
print("Entrada raw:", RAW_DIR)
print("Salida canónica:", OUT_DIR)
print("Reportes:", LOG_DIR)

Raíz del proyecto: C:\Users\hazar\Desktop\PROYECTO
Entrada raw: C:\Users\hazar\Desktop\PROYECTO\01_recoleccion_manual\03_descargas_raw
Salida canónica: C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico
Reportes: C:\Users\hazar\Desktop\PROYECTO\outputs


## 2. Utilidades de lectura, limpieza y normalización

In [2]:
MARCADORES_VACIOS = {
    "", "nan", "none", "null", "na", "n/a", "n.d.", "n.d", "n/d", "nd", "s/d",
    "sin dato", "sin datos", "sin información", "no disponible", "not available", "not provided",
    "unknown", "no aplica", "-", "--",
}

DOI_RE = re.compile(r"10\.\d{4,9}/[^\s\"<>]+", flags=re.IGNORECASE)
ISSN_RE = re.compile(r"\b\d{4}[- ]?\d{3}[\dXx]\b")
ISBN_RE = re.compile(r"(?<!\w)(?:97[89][-\s]?)?(?:\d[-\s]?){9}[\dXx](?!\w)")


def es_vacio(valor):
    if valor is None:
        return True
    try:
        if pd.isna(valor):
            return True
    except Exception:
        pass
    return str(valor).strip().casefold() in MARCADORES_VACIOS


def limpiar_texto(valor):
    """Limpia texto y convierte marcadores de ausencia a vacío."""
    if es_vacio(valor):
        return ""
    texto = str(valor).replace("\ufeff", "").strip()
    texto = re.sub(r"[\r\n\t]+", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    if texto.casefold() in MARCADORES_VACIOS:
        return ""
    return texto


def limpiar_df(df):
    """Limpia encabezados, columnas sin nombre, celdas y filas vacías."""
    df = df.copy()
    df.columns = [limpiar_texto(c) for c in df.columns]
    df = df.loc[:, [c for c in df.columns if c and not c.lower().startswith("unnamed")]]
    for col in df.columns:
        df[col] = df[col].map(limpiar_texto)
    if len(df.columns):
        df = df.loc[df.apply(lambda fila: any(limpiar_texto(v) for v in fila), axis=1)].reset_index(drop=True)
    return df


def normalizar_columna(nombre):
    texto = limpiar_texto(nombre).lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", "", texto)


def resolver_columna(df, candidatos):
    mapa = {normalizar_columna(col): col for col in df.columns}
    for candidato in candidatos:
        clave = normalizar_columna(candidato)
        if clave in mapa:
            return mapa[clave]
    return None


def serie_vacia(df):
    return pd.Series([""] * len(df), index=df.index, dtype="object")


def obtener_serie(df, candidatos):
    """Obtiene la primera columna existente de una lista de candidatos."""
    if isinstance(candidatos, str):
        candidatos = [candidatos]
    col = resolver_columna(df, candidatos)
    if col is None:
        return serie_vacia(df)
    return df[col].map(limpiar_texto)


def coalesce_columnas(df, candidatos):
    """Toma por fila el primer valor no vacío entre columnas candidatas."""
    resultado = serie_vacia(df)
    for candidato in candidatos:
        serie = obtener_serie(df, candidato)
        mascara = resultado.eq("") & serie.ne("")
        resultado.loc[mascara] = serie.loc[mascara]
    return resultado


def combinar_columnas(df, candidatos, separador=" | "):
    """Combina columnas no vacías sin duplicados exactos. Útil para pasar Afiliacion2 a Afiliacion1."""
    series = [obtener_serie(df, candidato) for candidato in candidatos]
    valores = []
    for i in range(len(df)):
        partes = []
        vistos = set()
        for serie in series:
            texto = limpiar_texto(serie.iloc[i])
            clave = texto.casefold()
            if texto and clave not in vistos:
                partes.append(texto)
                vistos.add(clave)
        valores.append(separador.join(partes))
    return pd.Series(valores, index=df.index, dtype="object")


def elegir_keywords(df, candidatos):
    """Usa la columna de keywords más directa; solo usa respaldos si la principal está vacía."""
    return coalesce_columnas(df, candidatos)


def detectar_encoding_csv(ruta):
    muestra = Path(ruta).read_bytes()[:200_000]
    for enc in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:
        try:
            muestra.decode(enc)
            return enc
        except UnicodeDecodeError:
            continue
    return "latin1"


def detectar_separador_csv(ruta, encoding):
    texto = Path(ruta).read_bytes()[:200_000].decode(encoding, errors="replace").lstrip("\ufeff")
    lineas = [l for l in texto.splitlines() if l.strip()]
    candidatos = [",", ";", "\t", "|"]
    if lineas:
        conteos = {sep: lineas[0].count(sep) for sep in candidatos}
        sep, n = max(conteos.items(), key=lambda x: x[1])
        if n > 0:
            return sep
    try:
        return csv.Sniffer().sniff(texto, delimiters=candidatos).delimiter
    except csv.Error:
        return ","


def leer_archivo(ruta):
    """Lee CSV/XLSX/XLS con dtype=str para no perder ceros ni formatos."""
    ruta = Path(ruta)
    sufijo = ruta.suffix.lower()
    if sufijo == ".csv":
        encoding = detectar_encoding_csv(ruta)
        sep = detectar_separador_csv(ruta, encoding)
        errores = []
        for enc in [encoding, "utf-8-sig", "utf-8", "cp1252", "latin1"]:
            for engine in ["c", "python"]:
                try:
                    df = pd.read_csv(
                        ruta,
                        sep=sep,
                        encoding=enc,
                        dtype=str,
                        keep_default_na=False,
                        na_values=[],
                        on_bad_lines="warn",
                        engine=engine,
                    )
                    return limpiar_df(df), {"encoding": enc, "separador": sep, "engine": engine}
                except Exception as e:
                    errores.append(str(e))
        raise RuntimeError(f"No se pudo leer {ruta.name}: {errores[-1] if errores else 'error desconocido'}")

    if sufijo in {".xlsx", ".xls"}:
        hojas = pd.read_excel(ruta, sheet_name=None, dtype=str, keep_default_na=False)
        for nombre_hoja, df in hojas.items():
            df = limpiar_df(df)
            if len(df):
                return df, {"hoja": nombre_hoja}
        return pd.DataFrame(), {"hoja": ""}

    raise ValueError(f"Extensión no soportada: {ruta.name}")


def normalizar_anio(valor):
    texto = limpiar_texto(valor)
    match = re.search(r"(19\d{2}|20\d{2})", texto)
    return int(match.group(1)) if match else ""


def normalizar_doi(valor):
    texto = limpiar_texto(valor)
    if not texto or texto == "0":
        return ""
    texto = re.sub(r"(?i)^doi\s*:\s*", "", texto)
    texto = re.sub(r"(?i)^https?://(dx\.)?doi\.org/", "", texto)
    texto = re.sub(r"(?i)^https?://doi\.org/", "", texto)
    texto = texto.replace(" ", "")
    match = DOI_RE.search(texto)
    if match:
        doi = match.group(0)
    elif texto.lower().startswith("10."):
        doi = texto
    else:
        return ""
    return doi.strip().strip(".;,)]}").lower()


def normalizar_issn(valor):
    """Extrae ISSN desde columnas ISSN. No usa eISSN."""
    texto = limpiar_texto(valor)
    if not texto or texto == "0":
        return ""
    encontrados = ISSN_RE.findall(texto.replace(" ", ""))
    salida, vistos = [], set()
    for item in encontrados:
        limpio = item.upper().replace("-", "")
        if len(limpio) == 8:
            formateado = f"{limpio[:4]}-{limpio[4:]}"
            if formateado not in vistos:
                salida.append(formateado)
                vistos.add(formateado)
    return "; ".join(salida)


def normalizar_isbn(valor):
    texto = limpiar_texto(valor)
    if not texto or texto == "0":
        return ""
    encontrados = ISBN_RE.findall(texto)
    salida, vistos = [], set()
    for item in encontrados:
        item = re.sub(r"\s+", "", item).strip(".;,")
        clave = re.sub(r"[^0-9Xx]", "", item).upper()
        if len(clave) in {10, 13} and clave not in vistos:
            salida.append(item)
            vistos.add(clave)
    return "; ".join(salida)


def limpiar_url(valor):
    texto = limpiar_texto(valor)
    if not texto or texto == "0":
        return ""
    return texto if texto.lower().startswith(("http://", "https://")) else ""


def normalizar_area(valor):
    texto = re.sub(r"[^A-Z]", "", limpiar_texto(valor).upper())
    alias = {"CC": "CC", "IA": "IA", "ISBD": "ISBD", "TC": "TC", "SIAV": "SIAV", "SIAAV": "SIAV", "RS": "RS"}
    return alias.get(texto, "")


def inferir_area_desde_archivo(ruta):
    tokens = [t for t in re.split(r"[^a-z0-9]+", Path(ruta).stem.lower()) if t]
    for token in tokens:
        area = normalizar_area(token)
        if area in AREAS_VALIDAS:
            return area
    return ""


def limpiar_autores_scopus(valor):
    texto = limpiar_texto(valor)
    texto = re.sub(r"\s*\(\d+\)", "", texto)
    texto = re.sub(r"\s+;", ";", texto)
    texto = re.sub(r";\s+", "; ", texto)
    return texto.strip()


def limpiar_autores_ev(valor):
    texto = limpiar_texto(valor)
    texto = re.sub(r"\s*\(\d+(?:\s*,\s*\d+)*\)", "", texto)
    texto = re.sub(r"\s+;", ";", texto)
    texto = re.sub(r";\s+", "; ", texto)
    return texto.strip()

## 3. Mapeos por fuente

Cada mapeo usa columnas candidatas en orden de prioridad. En `Keywords` se toma la columna más directa y solo se usa respaldo si está vacía.

In [3]:
MAPEOS = {
    "acm": {
        "indice": ["indice"], "Titulo": ["Titulo"], "Año": ["Año"], "Autor_norm": ["Autor_norm"],
        "Afiliacion1": ["Afiliacion1", "Afiliacion2"], "Afiliacion1_combinar": True,
        "ISBN": ["ISBN"], "ISSN": ["ISSN"], "Doi": ["Doi", "DOI"], "URL": ["URL"],
        "Area": ["Area"], "Keywords": ["Keywords"], "Abstract": ["Abstract"],
    },
    "sciencedirect": {
        "indice": ["indice"], "Titulo": ["Titulo"], "Año": ["Año"], "Autor_norm": ["Autor_norm"],
        "Afiliacion1": ["Afiliacion1", "Afiliacion2"], "Afiliacion1_combinar": True,
        "ISBN": ["ISBN"], "ISSN": ["ISSN"], "Doi": ["Doi", "DOI"], "URL": ["URL"],
        "Area": ["Area"], "Keywords": ["Keywords"], "Abstract": ["Abstract"],
    },
    "ebsco": {
        "Titulo": ["title", "Title"], "Año": ["publicationDate", "coverDate", "pubdate", "year"],
        "Autor_norm": ["contributors", "authors", "Authors"],
        "Afiliacion1": ["Afiliacion1", "AuthorAffiliation", "Affiliations"],
        "ISBN": ["isbns", "ISBN", "ISBNs"], "ISSN": ["issns", "ISSN", "issn"],
        "Doi": ["doi", "doids", "DOI"], "URL": ["plink", "url", "URL"],
        "Keywords": ["subjects", "identifierKeywords", "keywords", "Keywords"], "Abstract": ["abstract", "Abstract"],
    },
    "engineering_village": {
        "Titulo": ["Title"], "Año": ["Publication year", "Issue date"], "Autor_norm": ["Author"],
        "Afiliacion1": ["Author affiliation"], "ISBN": ["ISBN13", "ISBN"], "ISSN": ["ISSN"],
        "Doi": ["DOI"], "URL": [],
        "Keywords": ["Uncontrolled terms", "Controlled/Subject terms", "Main Heading"], "Abstract": ["Abstract"],
    },
    "ieee_xplore": {
        "Titulo": ["Document Title"], "Año": ["Publication Year", "Issue Date", "Online Date"],
        "Autor_norm": ["Authors"], "Afiliacion1": ["Author Affiliations"],
        "ISBN": ["ISBNs", "ISBN"], "ISSN": ["ISSN"], "Doi": ["DOI"], "URL": ["PDF Link"],
        "Keywords": ["Author Keywords", "IEEE Terms", "Mesh_Terms"], "Abstract": ["Abstract"],
    },
    "proquest": {
        "Titulo": ["Title"], "Año": ["year", "pubdate", "elecPubDate"], "Autor_norm": ["Authors"],
        "Afiliacion1": ["AuthorAffiliation"], "ISBN": ["ISBN", "ISBNs"], "ISSN": ["issn", "ISSN"],
        "Doi": ["digitalObjectIdentifier", "DOI", "doi"], "URL": ["DocumentURL"],
        "Keywords": ["identifierKeywords", "subjects", "subjectTerms"], "Abstract": ["Abstract"],
    },
    "scopus": {
        "Titulo": ["Title"], "Año": ["Year"], "Autor_norm": ["Author full names", "Authors"],
        "Afiliacion1": ["Affiliations", "Authors with affiliations"], "ISBN": ["ISBN"], "ISSN": ["ISSN"],
        "Doi": ["DOI"], "URL": ["Link"],
        "Keywords": ["Author Keywords", "Index Keywords"], "Abstract": ["Abstract"],
    },
    "web_of_science": {
        "Titulo": ["Article Title"], "Año": ["Publication Year", "Publication Date"],
        "Autor_norm": ["Author Full Names", "Authors"], "Afiliacion1": ["Affiliations", "Addresses"],
        "ISBN": ["ISBN"], "ISSN": ["ISSN"],  # No se agrega eISSN.
        "Doi": ["DOI"], "URL": ["DOI Link"],
        "Keywords": ["Author Keywords", "Keywords Plus"], "Abstract": ["Abstract"],
    },
}


def mapear_generico(df, ruta, fuente):
    df = limpiar_df(df)
    cfg = MAPEOS[fuente]
    out = pd.DataFrame(index=df.index)

    out["indice"] = coalesce_columnas(df, cfg.get("indice", [])) if cfg.get("indice") else ""
    out["Titulo"] = coalesce_columnas(df, cfg.get("Titulo", []))
    out["Año"] = coalesce_columnas(df, cfg.get("Año", []))
    out["Autor_norm"] = coalesce_columnas(df, cfg.get("Autor_norm", []))

    if cfg.get("Afiliacion1_combinar"):
        out["Afiliacion1"] = combinar_columnas(df, cfg.get("Afiliacion1", []))
    else:
        out["Afiliacion1"] = coalesce_columnas(df, cfg.get("Afiliacion1", []))

    out["Afiliacion2"] = ""
    out["ISBN"] = coalesce_columnas(df, cfg.get("ISBN", []))
    out["ISSN"] = coalesce_columnas(df, cfg.get("ISSN", []))
    out["Doi"] = coalesce_columnas(df, cfg.get("Doi", []))
    out["URL"] = coalesce_columnas(df, cfg.get("URL", []))
    out["Area"] = coalesce_columnas(df, cfg.get("Area", [])) if cfg.get("Area") else ""
    out["Keywords"] = elegir_keywords(df, cfg.get("Keywords", []))
    out["Abstract"] = coalesce_columnas(df, cfg.get("Abstract", []))
    out["Subarea"] = ""

    area_archivo = inferir_area_desde_archivo(ruta)
    out["Area"] = out["Area"].map(normalizar_area)
    if area_archivo:
        out.loc[out["Area"].eq(""), "Area"] = area_archivo

    out["Año"] = out["Año"].map(normalizar_anio)
    out["Doi"] = out["Doi"].map(normalizar_doi)
    out["ISBN"] = out["ISBN"].map(normalizar_isbn)
    out["ISSN"] = out["ISSN"].map(normalizar_issn)
    out["URL"] = out["URL"].map(limpiar_url)

    if fuente == "scopus":
        out["Autor_norm"] = out["Autor_norm"].map(limpiar_autores_scopus)
    elif fuente == "engineering_village":
        out["Autor_norm"] = out["Autor_norm"].map(limpiar_autores_ev)

    for col in COLUMNAS_CANONICAS:
        if col not in out.columns:
            out[col] = ""
        out[col] = out[col].map(limpiar_texto)

    out["Afiliacion2"] = ""
    out["Subarea"] = ""
    return out[COLUMNAS_CANONICAS]


# Funciones por fuente, explícitas para la defensa del proceso.
def mapear_acm(df, ruta):
    return mapear_generico(df, ruta, "acm")


def mapear_ebsco(df, ruta):
    return mapear_generico(df, ruta, "ebsco")


def mapear_engineering_village(df, ruta):
    return mapear_generico(df, ruta, "engineering_village")


def mapear_ieee_xplore(df, ruta):
    return mapear_generico(df, ruta, "ieee_xplore")


def mapear_proquest(df, ruta):
    return mapear_generico(df, ruta, "proquest")


def mapear_sciencedirect(df, ruta):
    return mapear_generico(df, ruta, "sciencedirect")


def mapear_scopus(df, ruta):
    return mapear_generico(df, ruta, "scopus")


def mapear_web_of_science(df, ruta):
    return mapear_generico(df, ruta, "web_of_science")


MAPEADORES = {
    "acm": mapear_acm,
    "ebsco": mapear_ebsco,
    "engineering_village": mapear_engineering_village,
    "ieee_xplore": mapear_ieee_xplore,
    "proquest": mapear_proquest,
    "sciencedirect": mapear_sciencedirect,
    "scopus": mapear_scopus,
    "web_of_science": mapear_web_of_science,
}

## 4. Procesamiento y generación de los 8 archivos canónicos

In [4]:
def encontrar_archivos_fuente(fuente):
    cfg = FUENTES[fuente]
    extensiones = {".csv", ".xlsx", ".xls"}
    archivos = []

    for carpeta_nombre in cfg["carpetas"]:
        carpeta = RAW_DIR / carpeta_nombre
        if carpeta.exists():
            archivos.extend([p for p in carpeta.iterdir() if p.is_file() and p.suffix.lower() in extensiones])

    # Respaldo: búsqueda recursiva por prefijo dentro de RAW_DIR.
    if not archivos:
        prefijos = tuple(p.lower() for p in cfg["prefijos_archivo"])
        for p in RAW_DIR.rglob("*"):
            if p.is_file() and p.suffix.lower() in extensiones and p.stem.lower().startswith(prefijos):
                archivos.append(p)

    return sorted(set(archivos), key=lambda p: p.name.lower())


def asegurar_modelo(df):
    df = df.copy()
    for col in COLUMNAS_CANONICAS:
        if col not in df.columns:
            df[col] = ""
    df = df[COLUMNAS_CANONICAS]
    for col in COLUMNAS_CANONICAS:
        df[col] = df[col].map(limpiar_texto)
    df["Año"] = df["Año"].map(normalizar_anio)
    df["Doi"] = df["Doi"].map(normalizar_doi)
    df["ISBN"] = df["ISBN"].map(normalizar_isbn)
    df["ISSN"] = df["ISSN"].map(normalizar_issn)
    df["URL"] = df["URL"].map(limpiar_url)
    df["Area"] = df["Area"].map(normalizar_area)
    df.loc[~df["Area"].isin(AREAS_VALIDAS), "Area"] = ""
    df["Afiliacion2"] = ""
    df["Subarea"] = ""
    return df[COLUMNAS_CANONICAS]


def crear_indices(df, fuente):
    prefijo = FUENTES[fuente]["prefijo_indice"]
    contadores = {}
    indices = []
    for area in df["Area"]:
        area = normalizar_area(area) or "SINAREA"
        contadores[area] = contadores.get(area, 0) + 1
        indices.append(f"{prefijo}_{area}_{contadores[area]:04d}")
    return pd.Series(indices, index=df.index, dtype="object")


def validar_canonico(df, fuente):
    if list(df.columns) != COLUMNAS_CANONICAS:
        raise ValueError(f"{fuente}: columnas incorrectas o fuera de orden.")
    if not df["Afiliacion2"].eq("").all():
        raise ValueError(f"{fuente}: Afiliacion2 debe estar vacía.")
    if not df["Subarea"].eq("").all():
        raise ValueError(f"{fuente}: Subarea debe estar vacía.")
    invalidas = sorted(set(df.loc[df["Area"].ne("") & ~df["Area"].isin(AREAS_VALIDAS), "Area"]))
    if invalidas:
        raise ValueError(f"{fuente}: áreas inválidas: {invalidas}")


def conteos_no_vacios(df):
    return {f"no_vacio_{col}": int(df[col].map(limpiar_texto).ne("").sum()) for col in COLUMNAS_CANONICAS}

## 5. Ejecutar conversión

Al ejecutar esta celda se generan los 8 archivos canónicos, uno por fuente. No se genera un consolidado único.

In [5]:
resumen = []
rutas_generadas = {}
salidas_por_fuente = {}

for fuente in FUENTES:
    archivos = encontrar_archivos_fuente(fuente)
    print(f"\n=== {fuente} ===")
    print(f"Archivos encontrados: {len(archivos)}")
    for archivo in archivos:
        print(" -", archivo.name)

    if not archivos:
        resumen.append({
            "fuente": fuente,
            "archivo": "",
            "estado": "sin_archivos",
            "filas_raw": 0,
            "filas_canonicas": 0,
            "columnas_raw": 0,
            "area_detectada": "",
            "encoding": "",
            "separador": "",
            "engine": "",
        })
        continue

    partes = []
    for archivo in archivos:
        df_raw, info = leer_archivo(archivo)
        df_can = MAPEADORES[fuente](df_raw, archivo)
        df_can = asegurar_modelo(df_can)
        validar_canonico(df_can, archivo.name)
        partes.append(df_can)

        fila_resumen = {
            "fuente": fuente,
            "archivo": archivo.name,
            "estado": "ok",
            "filas_raw": len(df_raw),
            "filas_canonicas": len(df_can),
            "columnas_raw": len(df_raw.columns),
            "area_detectada": inferir_area_desde_archivo(archivo),
            "encoding": info.get("encoding", ""),
            "separador": info.get("separador", ""),
            "engine": info.get("engine", ""),
        }
        fila_resumen.update(conteos_no_vacios(df_can))
        resumen.append(fila_resumen)

    df_fuente = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame(columns=COLUMNAS_CANONICAS)
    df_fuente = asegurar_modelo(df_fuente)

    if REGENERAR_INDICE and len(df_fuente):
        df_fuente["indice"] = crear_indices(df_fuente, fuente)

    validar_canonico(df_fuente, fuente)

    ruta_salida = OUT_DIR / FUENTES[fuente]["salida"]
    df_fuente.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

    if GUARDAR_XLSX:
        df_fuente.to_excel(ruta_salida.with_suffix(".xlsx"), index=False)

    rutas_generadas[fuente] = ruta_salida
    salidas_por_fuente[fuente] = df_fuente
    print(f"Salida: {ruta_salida.name} | filas: {len(df_fuente):,}")

resumen_df = pd.DataFrame(resumen)
ruta_resumen = LOG_DIR / "01_conversion_canonica_resumen.csv"
resumen_df.to_csv(ruta_resumen, index=False, encoding="utf-8-sig")

print("\nConversión terminada.")
print("Resumen guardado en:", ruta_resumen)
resumen_df


=== acm ===
Archivos encontrados: 6
 - acm_cc_depurado_cs_final_2024-03_2025-12.csv
 - acm_ia_depurado_cs_final_2024-03_2025-12.csv
 - acm_isbd_depurado_cs_final_2024-03_2025-12.csv
 - acm_rs_depurado_cs_final_2024-03_2025-12.csv
 - acm_siav_depurado_cs_final_2024-03_2025-12.csv
 - acm_tc_depurado_cs_final_2024-03_2025-12.csv
Salida: acm_canonico_2024-03_2025-12.csv | filas: 57

=== ebsco ===
Archivos encontrados: 6
 - ebsco_cc_depurado_cs_final_2024-03_2025-12.csv
 - ebsco_ia_depurado_cs_final_2024-03_2025-12.csv
 - ebsco_isbd_depurado_cs_final_2024-03_2025-12.csv
 - ebsco_rs_depurado_cs_final_2024-03_2025-12.csv
 - ebsco_siav_depurado_cs_final_2024-03_2025-12.csv
 - ebsco_tc_depurado_cs_final_2024-03_2025-12.csv
Salida: ebsco_canonico_2024-03_2025-12.csv | filas: 129

=== engineering_village ===
Archivos encontrados: 6
 - ev_cc_depurado_cs_final_2024-03_2025-12.csv
 - ev_ia_depurado_cs_final_2024-03_2025-12.csv
 - ev_isbd_depurado_cs_final_2024-03_2025-12.csv
 - ev_rs_depurado_cs_fi

,fuente,archivo,estado,filas_raw,filas_canonicas,columnas_raw,area_detectada,encoding,separador,engine,no_vacio_indice,no_vacio_Titulo,no_vacio_Año,no_vacio_Autor_norm,no_vacio_Afiliacion1,no_vacio_Afiliacion2,no_vacio_ISBN,no_vacio_ISSN,no_vacio_Doi,no_vacio_URL,no_vacio_Area,no_vacio_Subarea,no_vacio_Keywords,no_vacio_Abstract
0,acm,acm_cc_depurado_cs_final_2024-03_2025-12.csv,ok,6,6,14,CC,utf-8-sig,",",c,6,6,6,6,6,0,5,1,6,6,6,0,6,6
1,acm,acm_ia_depurado_cs_final_2024-03_2025-12.csv,ok,11,11,14,IA,utf-8-sig,",",c,11,11,11,11,11,0,9,2,11,11,11,0,11,11
2,acm,acm_isbd_depurado_cs_final_2024-03_2025-12.csv,ok,12,12,14,ISBD,utf-8-sig,",",c,12,12,12,12,12,0,9,3,12,12,12,0,11,11
3,acm,acm_rs_depurado_cs_final_2024-03_2025-12.csv,ok,10,10,14,RS,utf-8-sig,",",c,10,10,10,10,10,0,8,2,10,10,10,0,9,9
4,acm,acm_siav_depurado_cs_final_2024-03_2025-12.csv,ok,7,7,14,SIAV,utf-8-sig,",",c,7,7,7,7,7,0,5,2,7,7,7,0,7,7
5,acm,acm_tc_depurado_cs_final_2024-03_2025-12.csv,ok,11,11,14,TC,utf-8-sig,",",c,11,11,11,11,11,0,10,1,11,11,11,0,10,10
6,ebsco,ebsco_cc_depurado_cs_final_2024-03_2025-12.csv,ok,15,15,39,CC,utf-8-sig,",",c,0,15,15,15,15,0,0,15,15,15,15,0,15,15
7,ebsco,ebsco_ia_depurado_cs_final_2024-03_2025-12.csv,ok,32,32,39,IA,utf-8-sig,",",c,0,32,32,32,32,0,0,32,32,32,32,0,30,32
8,ebsco,ebsco_isbd_depurado_cs_final_2024-03_2025-12.csv,ok,9,9,39,ISBD,utf-8-sig,",",c,0,9,9,9,9,0,0,9,9,9,9,0,7,9
9,ebsco,ebsco_rs_depurado_cs_final_2024-03_2025-12.csv,ok,32,32,39,RS,utf-8-sig,",",c,0,32,32,32,32,0,0,32,32,32,32,0,30,32


## 6. Validación final

In [6]:
validaciones = []
for fuente, ruta in rutas_generadas.items():
    df_check = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    validaciones.append({
        "fuente": fuente,
        "archivo": ruta.name,
        "filas": len(df_check),
        "columnas": len(df_check.columns),
        "columnas_correctas": list(df_check.columns) == COLUMNAS_CANONICAS,
        "afiliacion2_vacia": df_check["Afiliacion2"].eq("").all(),
        "subarea_vacia": df_check["Subarea"].eq("").all(),
        "areas": ", ".join(sorted([a for a in df_check["Area"].unique() if a])),
    })

validaciones_df = pd.DataFrame(validaciones)
display(validaciones_df)

esperados = set(FUENTES.keys())
generados = set(rutas_generadas.keys())
faltantes = sorted(esperados - generados)

print("\nArchivos canónicos generados:")
for fuente, ruta in rutas_generadas.items():
    print(f"- {fuente}: {ruta}")

if faltantes:
    print("\nFuentes faltantes:", faltantes)

assert not faltantes, "Faltó generar una o más fuentes. Revisa carpetas y nombres de archivos raw."
assert validaciones_df["columnas_correctas"].all(), "Alguna salida no tiene las columnas canónicas exactas."
assert validaciones_df["afiliacion2_vacia"].all(), "Alguna salida tiene Afiliacion2 no vacía."
assert validaciones_df["subarea_vacia"].all(), "Alguna salida tiene Subarea no vacía."

,fuente,archivo,filas,columnas,columnas_correctas,afiliacion2_vacia,subarea_vacia,areas
0,acm,acm_canonico_2024-03_2025-12.csv,57,14,True,True,True,"CC, IA, ISBD, RS, SIAV, TC"
1,ebsco,ebsco_canonico_2024-03_2025-12.csv,129,14,True,True,True,"CC, IA, ISBD, RS, SIAV, TC"
2,engineering_village,engineering_village_canonico_2024-03_2025-12.csv,142,14,True,True,True,"CC, IA, ISBD, RS, SIAV, TC"
3,ieee_xplore,ieee_xplore_canonico_2024-03_2025-12.csv,158,14,True,True,True,"CC, IA, ISBD, RS, SIAV, TC"
4,proquest,proquest_canonico_2024-03_2025-12.csv,19,14,True,True,True,"CC, IA, ISBD, RS, SIAV, TC"
5,sciencedirect,sciencedirect_canonico_2024-03_2025-12.csv,43,14,True,True,True,"CC, IA, ISBD, RS, SIAV, TC"
6,scopus,scopus_canonico_2024-03_2025-12.csv,561,14,True,True,True,"CC, IA, ISBD, RS, SIAV, TC"
7,web_of_science,web_of_science_canonico_2024-03_2025-12.csv,362,14,True,True,True,"CC, IA, ISBD, RS, SIAV, TC"



Archivos canónicos generados:
- acm: C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\acm_canonico_2024-03_2025-12.csv
- ebsco: C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\ebsco_canonico_2024-03_2025-12.csv
- engineering_village: C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\engineering_village_canonico_2024-03_2025-12.csv
- ieee_xplore: C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\ieee_xplore_canonico_2024-03_2025-12.csv
- proquest: C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\proquest_canonico_2024-03_2025-12.csv
- sciencedirect: C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\sciencedirect_canonico_2024-03_2025-12.csv
- scopus: C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\scopus_canonico_2024-03_2025-12.csv
- web_of_science: C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\web_of_science_canonico_2024-03_2025-12.csv


## 7. Vista previa opcional por fuente

In [7]:
for fuente, df in salidas_por_fuente.items():
    print(f"\n--- {fuente} | filas: {len(df):,} ---")
    display(df.head(3))


--- acm | filas: 57 ---


,indice,Titulo,Año,Autor_norm,Afiliacion1,Afiliacion2,ISBN,ISSN,Doi,URL,Area,Subarea,Keywords,Abstract
0,ACM_CC_0001,The Computational Power of Distributed Shared-Memory Models with Bounded-Size Registers,2024,"Delporte-Gallet, Carole; Fauconnier, Hugues; Fraigniaud, Pierre; Rajsbaum, Sergio; Travers, Corentin","Carole Delporte-Gallet: IRIF, University Paris Cité, CNRS, Paris, France; Hugues Fauconnier: IRIF, University Paris ...",,979-8-4007-0668-4,,10.1145/3662158.3662789,https://doi.org/10.1145/3662158.3662789,CC,,shared memory; wait-freedom; t-resiliency; approximate agreement; space complexity,The celebrated Asynchronous Computability Theorem of Herlihy and Shavit (JACM 1999) provided a topological character...
1,ACM_CC_0002,Asynchronous Fault-Tolerant Language Decidability for Runtime Verification of Distributed Systems,2025,"Castañeda, Armando; Rodríguez, Gilde Valeria","Armando Castañeda: Instituto de Matemáticas, Universidad Nacional Autónoma de México, Mexico City, México; Gilde Val...",,979-8-4007-1885-4,,10.1145/3732772.3733514,https://doi.org/10.1145/3732772.3733514,CC,,concurrent algorithms; distributed runtime verification; distributed systems; eventual consistency fault-tolerance; ...,Implementing correct distributed systems is an error-prone task. Runtime Verification (RV) offers a lightweight form...
2,ACM_CC_0003,Strong Linearizability using Primitives with Consensus Number 2,2024,"Attiya, Hagit; Castañeda, Armando; Enea, Constantin","Hagit Attiya: Technion, Department of Computer Science, Israel; Armando Castañeda: Universidad Nacional Autónoma de ...",,979-8-4007-0668-4,,10.1145/3662158.3662790,https://doi.org/10.1145/3662158.3662790,CC,,concurrency; consensus numbers; linearizability; lock-freedom; shared memory; strong linearizability; wait-freedom,A powerful tool for designing complex concurrent programs is through composition with object implementations from lo...



--- ebsco | filas: 129 ---


,indice,Titulo,Año,Autor_norm,Afiliacion1,Afiliacion2,ISBN,ISSN,Doi,URL,Area,Subarea,Keywords,Abstract
0,EBSCO_CC_0001,A time-domain SGFD-FK hybrid method for 2D teleseismic elastic wave modeling and inversion.,2024,"del Valle-Rosales, Mauricio ; Chávez-García, Francisco José","Mauricio del Valle-Rosales — Posgrado en Ciencias de la Tierra, Universidad Nacional Autónoma de México, Mexico City...",,,1895-6572,10.1007/s11600-024-01335-1,https://research.ebsco.com/plink/ed52599a-17ab-36e9-a399-2905f0913fa0,CC,,Finite difference method ; Elastic waves ; Theory of wave motion ; Seismic tomography ; Seismology,Full waveform inversion (FWI) has proved to be a reliable tool for high-resolution imaging of lithospheric structure...
1,EBSCO_CC_0002,Simulation of the behavior of fine and gross motor skills of an individual with motor disabilities.,2024,"Sánchez-Torres, Karla K. ; Rodríguez-Romo, Suemi","Karla K. Sánchez-Torres — Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, UNAM, Escolar 3000, C...",,,0941-0643,10.1007/s00521-024-10267-2,https://research.ebsco.com/plink/4731e3a6-6fa0-32c6-bcd9-a4b831322ca6,CC,,Artificial neural networks ; Fine motor ability ; Motor ability ; Central nervous system ; Reinforcement learning ; ...,We have developed a neural network model that imitates the central nervous system's control of motor sensors (Sánche...
2,EBSCO_CC_0003,Solution Algorithms for the Capacitated Location Tree Problem with Interconnections.,2025,"Mendoza-Andrade, Nidia ; Ruiz-y-Ruiz, Efrain ; Rodriguez-Romo, Suemi","Nidia Mendoza-Andrade — Facultad de Estudios Superiores Cuautitlán, UNAM/Universidad Nacional Autónoma de México, Ci...",,,1999-4893,10.3390/a18010050,https://research.ebsco.com/plink/4a804b41-97fd-3b10-8763-643887887c37,CC,,Combinatorial optimization ; Metaheuristic algorithms ; Satisfaction ; Distribution costs ; Consumers,"This paper addresses the Capacitated Location Tree Problem with Interconnections, a new combinatorial optimization p..."



--- engineering_village | filas: 142 ---


...

,indice,Titulo,Año,Autor_norm,Afiliacion1,Afiliacion2,ISBN,ISSN,Doi,URL,Area,Subarea,Keywords,Abstract
0,SD_CC_0001,An efficient algorithm for identifying rainbow ortho-convex 4-sets in k-colored point sets,2025,"Flores-Peñaloza, David; Lopez, Mario A.; Marín, Nestaly; Orden, David","David Flores-Peñaloza: Departamento de Matemáticas, Facultad de Ciencias, Universidad Nacional Autónoma de México, M...",,,0020-0190,10.1016/j.ipl.2024.106551,https://doi.org/10.1016/j.ipl.2024.106551,CC,,Orthogonal convexity; Colored point sets; Separability; Algebraic computation tree model,"Let P be a k-colored set of n points in the plane, 4 <= k <= n. We study the problem of deciding if P contains a sub..."
1,SD_CC_0002,A novel method for ECG signal morphology analysis using tortuosity estimation,2024,"Pacheco González, Luis Eduardo; Torres Guzmán, Didier; Barbará-Morales, Eduardo","Luis Eduardo Pacheco González: Universidad Nacional Autónoma de México, Ciudad Universitaria, Coyoacán, 04510 Ciudad...",,,1746-8094,10.1016/j.bspc.2024.106772,https://doi.org/10.1016/j.bspc.2024.106772,CC,,Electrocardiogram; Morphology analysis; Tortuosity; Slope chain code,Objective: The present work aims to evaluate an electrocardiogram (ECG) signal with a new algorithm based on the tor...
2,SD_CC_0003,CUDA code to generate computational models and predict mechanical properties for metallic surface nanocoatings,2024,"Bedolla-Hernández, M.; Sánchez-Ruiz, F.J.; Rosano-Ortega, G.; Bedolla-Hernández, J.; Schabes-Retchkiman, P.S.; Vega-...","M. Bedolla-Hernández: Tecnológico Nacional de México/I. T. Apizaco, Apizaco, Tlaxcala, México | F.J. Sánchez-Ruiz: U...",,,2665-9638,10.1016/j.simpa.2024.100645,https://doi.org/10.1016/j.simpa.2024.100645,CC,,CUDA programming; Metallic surface nanocoating; FEM simulation; Predictive models; Mechanical surface properties,"The article presents an open-access code, written in CUDA and C++ programming language, applicable for generating co..."



--- scopus | filas: 561 ---


,indice,Titulo,Año,Autor_norm,Afiliacion1,Afiliacion2,ISBN,ISSN,Doi,URL,Area,Subarea,Keywords,Abstract
0,SCOPUS_CC_0001,"Time integration scheme for nonlinear structural dynamics, FAM, including structural vibration control",2025,"Patlán, Carlos M.; Hernández-Barrios, Hugo; Huergo, Iván F.; Domínguez-Mota, Francisco","School of Civil Engineering, Universidad Nacional Autónoma de México, Mexico City, Mexico; School of Civil Engineeri...",,,0045-7949,10.1016/j.compstruc.2025.107645,https://www.scopus.com/pages/publications/85215411373?origin=resultslist,CC,,Force Analogy Method; Inelastic analysis; Nonlinear control systems; State space formulation; Time domain integration,"In this study, a method for the integration of the equation of motion for the inelastic analysis of structures utili..."
1,SCOPUS_CC_0002,Physics-Informed Neural Networks for Higher-Order Nonlinear Schrödinger Equations: Soliton Dynamics in External Pote...,2025,"Serkin, Leonid; Belyaeva, Tatyana L.","Facultad de Ciencias, Universidad Nacional Autónoma de México, Ciudad Universitaria, Circuito Exterior s/n, Coyoacán...",,,2227-7390,10.3390/math13111882,https://www.scopus.com/pages/publications/105007771323?origin=resultslist,CC,,external potentials; NLSE; PINN; solitons,This review summarizes the application of physics-informed neural networks (PINNs) for solving higher-order nonlinea...
2,SCOPUS_CC_0003,Multipopulation mathematical modeling of vaccination campaign of COVID-19 in Cuba,2025,"Guinovart-Díaz, Raúl; Morales-Lezca, Wilfredo; Abelló-Ugalde, Isidro; Bravo-Castillero, Julián; Vajravelu, Kuppalapa...","Facultad de Matemática y Computación, Universidad de La Habana, La Habana, Cuba; Centro de Estudios para el Perfecci...",,,0228-6203,10.1080/02286203.2025.2457762,https://www.scopus.com/pages/publications/85217186513?origin=resultslist,CC,,COVID-19; Dynamical systems; epidemiology; Multi-Population Approach; Pandemic Simulation; SIR Model; Transmission D...,Cuba is facing a severe COVID-19 outbreak despite its efforts to contain the virus and vaccinate its population. We ...



--- web_of_science | filas: 362 ---


,indice,Titulo,Año,Autor_norm,Afiliacion1,Afiliacion2,ISBN,ISSN,Doi,URL,Area,Subarea,Keywords,Abstract
0,WOS_CC_0001,On the Approximation of the Entire Pareto Front of a Constrained Multi-objective Optimization Problem,2025,"Rodriguez-Fernandez, Angel E.; Hernandez Castellanos, Carlos Ignacio; Schutze, Oliver",Instituto Politecnico Nacional - Mexico; CINVESTAV - Centro de Investigacion y de Estudios Avanzados del Instituto P...,,978-981-96-3537-5; 978-981-96-3538-2,0302-9743,10.1007/978-981-96-3538-2_7,,CC,,Multi-objective optimization; Constraint handling; Archiving,"So far, many constraint-handling techniques (CHTs) exist that allow specialized multi-objective evolutionary algorit..."
1,WOS_CC_0002,Computational Geodynamics Laboratory: 15 years of numerical modelling and high-performance computing at the Centro d...,2024,"Manea, Marina; Manea, Vlad C.",Universidad Nacional Autonoma de Mexico,,,1026-8774,10.22201/cgeo.20072902e.2024.1.111773,,CC,,Computational geodynamics; numerical modeling; supercomputing,The rapid development of computer technology in the last decades provided scientists with new opportunities to expan...
2,WOS_CC_0003,A parallel steganography algorithm using a new secret key generation,2024,"Al-Jarah, Ahmed Imad Hammoodi; Ortega Arjona, Jorge Luis",Universidad Nacional Autonoma de Mexico; Universidad Nacional Autonoma de Mexico,,979-8-3315-4091-3; 979-8-3315-4090-6,,10.1109/uemcon62879.2024.10754748,,CC,,Hiding data; Image steganography; Processing time improvement; Parallel steganography; Secret key steganography; Ste...,"Computation has mostly used parallelism for high-performance computation, and multicore processors have gained popul..."
